# nodes.embed_ads

Embed raw job ad text directly using the configured embedding model.
No LLM summarisation needed.

1. Loads raw ad text (title + description) from Adzuna DuckDB.
2. Reads prompt support config from `embed_models.toml` for the model:
   - `query_prefix`: fixed string prepended unconditionally
   - `query_prompt_name`: named prompt passed to SentenceTransformer
   - `supports_prompt`: whether the model accepts a custom task instruction
3. Embeds in chunks with resume support via ResultStore.
4. Saves embeddings to `store/pipeline/{run_name}/embed_ads/`.

Node variables:
- `embedding_model` (global): Model key from embed_models.toml
- `embed_task_prompt` (per-node): Custom task instruction (only used if
  model has `supports_prompt = true`)
- `max_concurrent_chunks` (per-node): Max concurrent embedding chunks
- `run_name` (global): Pipeline run name

In [ ]:
#|default_exp embed_ads
#|export_as_func true

In [ ]:
#|set_func_signature
async def main(ctx, print, ad_ids: "np.ndarray") -> {
    'ad_ids': list[int]
}:
    """Embed raw job ad text directly."""
    ...


Retrieve input arguments

In [ ]:
from dev_utils import *
run_name = 'test_local'
set_node_func_args('embed_ads', run_name=run_name)
show_node_vars('embed_ads', run_name=run_name)

# Function body

## Read node variables

In [ ]:
#|export
import json
import time

import numpy as np
import pandas as pd

from ai_index import const
from ai_index.utils import aembed, get_ads_by_id
from ai_index.utils._model_config import _load_model_config
from ai_index.utils.batch import run_batched
from ai_index.utils.result_store import ResultStore

In [ ]:
#|export
run_name = ctx.vars["run_name"]
embedding_model = ctx.vars["embedding_model"]
sbatch_cache = ctx.vars["sbatch_cache"]
sbatch_time = ctx.vars["sbatch_time"]
embed_task_prompt = ctx.vars["embed_task_prompt"]
chunk_size = ctx.vars["chunk_size"]
max_concurrent_chunks = ctx.vars["max_concurrent_chunks"]
duckdb_memory_limit = ctx.vars["duckdb_memory_limit"]

output_dir = const.pipeline_store_path / run_name / "embed_ads"
output_dir.mkdir(parents=True, exist_ok=True)

## Read prompt config from embed_models.toml

In [ ]:
#|export
_, model_cfg = _load_model_config(const.embed_models_config_path, embedding_model)
query_prefix = model_cfg.get("query_prefix", "")
query_prompt_name = model_cfg.get("query_prompt_name", None)
supports_prompt = model_cfg.get("supports_prompt", False)

# Determine the prompt kwarg to pass to aembed()
embed_prompt_kwargs = {}
if supports_prompt and embed_task_prompt:
    embed_prompt_kwargs["prompt"] = embed_task_prompt
    print(f"embed_ads: using custom task prompt: {embed_task_prompt[:80]}...")
elif query_prompt_name:
    embed_prompt_kwargs["prompt_name"] = query_prompt_name
    print(f"embed_ads: using named prompt: {query_prompt_name}")

if query_prefix:
    print(f"embed_ads: applying query prefix: {query_prefix!r}")

## Load raw ad texts

In [ ]:
#|export
ad_ids_list = [int(i) for i in ad_ids]
ads_table = get_ads_by_id(ad_ids_list, columns=["title", "description"], memory_limit=duckdb_memory_limit)
ads_df = ads_table.to_pandas().set_index("id")

# Build texts: query_prefix + title + ". " + description
texts_by_id = {}
for ad_id in ad_ids_list:
    row = ads_df.loc[ad_id]
    title = str(row["title"] or "")
    desc = str(row["description"] or "")
    text = f"{title}. {desc}"
    texts_by_id[ad_id] = query_prefix + text

# Maintain order matching ad_ids
ordered_ids = [i for i in ad_ids_list if i in texts_by_id]
ordered_texts = [texts_by_id[i] for i in ordered_ids]

n_total = len(ordered_ids)
print(f"embed_ads: loaded {n_total} ad texts")
if n_total > 0:
    sample = ordered_texts[0][:120]
    print(f"  sample: {sample}...")

## Embed in chunks

Embeds ads in chunks via `run_batched`, which handles resume, concurrency,
and retry. Each chunk calls `aembed()` and returns a DataFrame for the
ResultStore.

In [ ]:
#|export
texts_by_id = dict(zip(ordered_ids, ordered_texts))

_slurm_jobs = []

async def _work_fn(chunk_ids):
    chunk_texts = [texts_by_id[i] for i in chunk_ids]

    _sa = {}
    embeddings = await aembed(
        chunk_texts, model=embedding_model,
        cache=sbatch_cache, time=sbatch_time,
        slurm_accounting=_sa,
        **embed_prompt_kwargs,
    )
    if _sa:
        _slurm_jobs.append(_sa)

    return pd.DataFrame({
        "id": list(chunk_ids),
        "embedding": [row.astype(np.float32).tobytes() for row in embeddings],
        "error": [None] * len(chunk_ids),
    })

In [ ]:
#|export
db_path = output_dir / "embeddings.duckdb"
store = ResultStore(db_path, {
    "id": "BIGINT NOT NULL",
    "embedding": "BLOB NOT NULL",
    "error": "VARCHAR",
}, memory_limit=duckdb_memory_limit)

embed_meta = await run_batched(
    ordered_ids, store, _work_fn,
    batch_size=chunk_size,
    max_concurrent=max_concurrent_chunks,
    resume=True,
    node_name="embed_ads",
    print_fn=print,
)
store.close()
del store
embed_meta["slurm_jobs"] = _slurm_jobs
embed_meta["slurm_total_seconds"] = sum(j.get("elapsed_seconds", 0) for j in _slurm_jobs)

meta_path = output_dir / "embed_meta.json"
with open(meta_path, "w") as f:
    json.dump(embed_meta, f, indent=2)
print(f"embed_ads: wrote {const.rel(meta_path)}")

ad_ids = ordered_ids
ad_ids #|func_return_line